<a href="https://colab.research.google.com/github/sethkipsangmutuba/Database-Management-System/blob/main/Week_9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 9: Distributed Databases & Global Consistency

## 1. Introduction

As data volumes continue to grow and applications scale globally, modern systems increasingly rely on distributed databases. These systems store and manage data across multiple machines or geographic locations to ensure high availability, fault tolerance, and scalability. However, distributing data introduces complex challenges in maintaining consistency, managing replication, and coordinating between nodes. This week, we explore the foundations of distributed databases, sharding, replication models, the CAP and PACELC theorems, and consensus protocols such as Paxos and Raft.

## 2. What is a Distributed Database?

A Distributed Database System is a database that runs on multiple servers but appears to users as a single, unified system. It enables data to be stored and processed across different physical locations, connected via a network.

### 2.1 Characteristics of Distributed Databases

- **Transparency**: Users do not need to know where the data resides.  
- **Scalability**: Easily scale out by adding more nodes.  
- **Availability**: System continues functioning even if some nodes fail.  
- **Fault Tolerance**: Ability to recover from partial system failures.  
- **Autonomy**: Each node can operate independently.

## 3. Sharding and Data Partitioning

### 3.1 Sharding

Sharding is the process of partitioning a database horizontally by distributing rows of a table across different machines. Each shard contains a subset of the data.

Example: A customer table could be sharded by customer ID range:  
- Shard 1 → IDs 1–1000  
- Shard 2 → IDs 1001–2000

### 3.2 Advantages of Sharding

- Improved performance via parallelism  
- Better resource utilization  
- Facilitates large-scale horizontal scaling

### 3.3 Challenges of Sharding

- Rebalancing data when nodes are added/removed  
- Maintaining referential integrity across shards  
- Performing distributed joins or transactions

## 4. Replication Models

Replication is used to copy data across multiple nodes for fault tolerance and improved read performance.

### 4.1 Types of Replication

- **Master-Slave Replication**: One primary node handles writes; replicas serve reads.  
- **Master-Master Replication**: Multiple nodes can accept writes and sync changes.  
- **Multi-Leader and Leaderless**: Used in systems like DynamoDB or Cassandra.

### 4.2 Synchronous vs Asynchronous Replication

- **Synchronous**: Writes are committed to all replicas before acknowledgment. Ensures strong consistency but increases latency.  
- **Asynchronous**: Writes are acknowledged once the primary node processes them. Faster but may result in stale reads.

## 5. Consistency Models

Distributed databases must manage how updates are seen across nodes. There are different consistency models based on system design trade-offs.

### 5.1 Strong Consistency

Guarantees that all nodes see the same data at the same time after an update.  
**Example**: Google Spanner  
- Pros: Predictable results  
- Cons: Higher latency and availability issues

### 5.2 Eventual Consistency

Guarantees that if no new updates are made, all nodes will eventually converge to the same value.  
**Example**: Amazon DynamoDB, Cassandra  
- Pros: Low latency and high availability  
- Cons: Temporary data inconsistency

### 5.3 Causal Consistency

Ensures that operations that are causally related are seen by all nodes in the same order.

## 6. CAP Theorem

Proposed by Eric Brewer, the CAP theorem states that a distributed system can provide at most two of the following three guarantees simultaneously:

- **Consistency (C)**: Every read gets the latest write or an error.  
- **Availability (A)**: Every request receives a response (without guarantee of the latest write).  
- **Partition Tolerance (P)**: The system continues to function despite network partitions.

### 6.1 Implications of CAP

- **CA**: Not possible in the presence of a partition.  
- **CP**: Sacrifices availability during a partition.  
- **AP**: Sacrifices consistency during a partition.

No distributed system can guarantee all three at the same time. In practice, systems choose two based on their requirements.

## 7. PACELC Theorem

The PACELC theorem extends CAP by addressing the trade-off even when there is no network partition.

PACELC stands for:  
If Partition $(P)$ happens, choose between Availability $(A)$ and Consistency $(C)$  
Else $(E)$, choose between Latency $(L)$ and Consistency $(C)$

### 7.1 Example Application

- **Amazon DynamoDB**: AP/EL → prioritizes availability during partitions, and low latency otherwise.  
- **Google Spanner**: CP/EC → prioritizes consistency at all times, even at the cost of availability or latency.

## 8. Distributed Consensus

In distributed systems, consensus is needed to ensure all nodes agree on a single value, such as the result of a transaction or the next leader in a replicated system.

### 8.1 Why Consensus is Hard

- Nodes may fail or become unreachable.  
- Messages may be lost or delayed.  
- Nodes may disagree due to outdated information.

### 8.2 Properties of Consensus Protocols

- **Termination**: Eventually, every correct process decides.  
- **Agreement**: All correct processes agree on the same value.  
- **Validity**: The decided value must have been proposed by a process.

## 9. Paxos Protocol

Paxos, developed by Leslie Lamport, is a family of protocols for solving consensus in a network of unreliable processors.

### 9.1 Basic Paxos Roles

- **Proposer**: Suggests a value  
- **Acceptor**: Accepts proposed values  
- **Learner**: Learns the agreed value

### 9.2 Steps in Paxos

- **Prepare Phase**: Proposer sends a proposal number to acceptors.  
- **Promise Phase**: Acceptors respond with a promise not to accept lower-numbered proposals.  
- **Accept Phase**: Proposer sends the actual value to be accepted.  
- **Accepted Phase**: Acceptors agree and notify learners.

### 9.3 Drawbacks of Paxos

- Complex to implement  
- Difficult to understand and debug  
- Requires multiple round trips, leading to high latency

## 10. Raft Protocol

Raft was introduced as a more understandable alternative to Paxos. It simplifies consensus by decomposing the problem into sub-problems and following a leader-based approach.

### 10.1 Raft Roles

- **Leader**: Handles all client interactions and log replication.  
- **Followers**: Passively replicate log entries.  
- **Candidates**: Attempt to become leader during elections.

### 10.2 Raft Components

- **Leader Election**: Nodes elect a leader using timeouts and votes.  
- **Log Replication**: Leader replicates entries to followers and waits for acknowledgment.  
- **Safety**: Ensures that committed logs are not lost and remain in order.

### 10.3 Advantages of Raft

- Easier to understand and implement than Paxos  
- Widely used in systems like etcd, Consul, and RethinkDB  
- Supports dynamic membership (adding/removing nodes)

## 11. Google Spanner: Global Consistency at Scale

Google Spanner is a globally distributed, strongly consistent database that addresses the limitations of traditional distributed systems.

### 11.1 Key Innovations

- **TrueTime API**: Combines clock synchronization and uncertainty to enable external consistency.  
- **Globally Synchronized Clocks**: Allows globally consistent reads and writes.  
- **Synchronous Replication**: Ensures strong consistency and high availability.

### 11.2 Trade-Offs in Spanner

- Higher latency due to consensus and clock synchronization  
- Greater infrastructure complexity (requires atomic clocks and GPS)

### 11.3 Lessons from Spanner

- Strong consistency is achievable at global scale  
- Requires precise coordination and advanced infrastructure  
- Demonstrates how theoretical models like Paxos can be applied in real-world, high-performance systems

## 12. Lab Activity: Simulating Distributed Transactions Using Raft

In this week's lab, students explore distributed consensus practically by simulating transactions coordinated using the Raft protocol.

**Objectives**:  
- Understand how leader election and log replication occur  
- Observe the impact of network partitions or node failures  
- Track how consensus is reached and maintained in a distributed system

Students will also compare Raft with Paxos and evaluate their use in real-world systems like etcd and Consul.

## 13. Reading Assignment: Google Spanner Whitepaper

Students are required to read the Google Spanner whitepaper, which provides an in-depth analysis of how Spanner achieves strong consistency, high availability, and global distribution.

**Key takeaways include**:  
- Design architecture  
- Use of synchronized clocks  
- Scalability techniques  
- Implementation of Paxos-based consensus

## 14. Summary

Distributed databases provide the scalability and resilience needed for modern, global applications. However, they bring complex challenges in maintaining consistency, availability, and fault tolerance.

**Key Takeaways**:
- Distributed systems use sharding and replication to scale and remain available.  
- The CAP theorem highlights trade-offs between consistency, availability, and partition tolerance.  
- PACELC extends CAP by adding latency vs consistency trade-offs even in the absence of failures.  
- Paxos and Raft are consensus algorithms that enable distributed systems to agree on values in the face of failures.  
- Google Spanner exemplifies a practical system that combines distributed consensus with synchronized clocks to achieve global consistency.


In [4]:
# Week 9: Distributed Databases & Global Consistency - Simulation in Python

import random
import time
from typing import List, Dict

# ========================
# Simulated Raft Consensus
# ========================

class Node:
    def __init__(self, node_id: int):
        self.node_id = node_id
        self.state = 'Follower'  # Can be Follower, Candidate, or Leader
        self.term = 0
        self.voted_for = None
        self.log = []
        self.commit_index = -1

    def __repr__(self):
        return f"Node{self.node_id}({self.state})"


class RaftCluster:
    def __init__(self, num_nodes: int = 3):
        self.nodes = [Node(i) for i in range(num_nodes)]
        self.leader = None
        self.current_term = 0

    def elect_leader(self):
        print("\n=== Leader Election Started ===")
        candidates = random.sample(self.nodes, len(self.nodes))
        for candidate in candidates:
            votes = 1  # Vote for self
            candidate.state = 'Candidate'
            candidate.term += 1
            candidate.voted_for = candidate.node_id

            for peer in self.nodes:
                if peer.node_id != candidate.node_id and (peer.voted_for is None or peer.voted_for == candidate.node_id):
                    votes += 1

            if votes > len(self.nodes) // 2:
                candidate.state = 'Leader'
                self.leader = candidate
                self.current_term = candidate.term
                print(f"Node {candidate.node_id} becomes Leader with {votes} votes (Term {self.current_term})")
                break

        if not self.leader:
            print("Election failed. No majority achieved.")

    def replicate_log(self, command: str):
        if not self.leader:
            print("No leader elected. Cannot replicate log.")
            return

        print(f"\nLeader (Node {self.leader.node_id}) is replicating command: '{command}'")
        self.leader.log.append((self.current_term, command))
        for node in self.nodes:
            if node.node_id != self.leader.node_id:
                node.log.append((self.current_term, command))
        print("Replication complete.")

    def show_logs(self):
        print("\n=== Logs of all Nodes ===")
        for node in self.nodes:
            print(f"Node {node.node_id} Log: {node.log}")

# ========================
# Simulated Sharding
# ========================

def shard_data(data: List[Dict], num_shards: int) -> Dict[int, List[Dict]]:
    shards = {i: [] for i in range(num_shards)}
    for record in data:
        shard_id = record['id'] % num_shards
        shards[shard_id].append(record)
    return shards

# ========================
# Simulated Replication
# ========================

def replicate_shards(shards: Dict[int, List[Dict]]) -> Dict[int, List[List[Dict]]]:
    replica_set = {}
    for shard_id, records in shards.items():
        replica_set[shard_id] = [records.copy() for _ in range(3)]  # 3 replicas per shard
    return replica_set

# ========================
# Simulated CAP/PACELC
# ========================

def cap_scenario(network_partition=False):
    print("\n=== CAP Theorem Scenario ===")
    if network_partition:
        print("Network partition detected.")
        print("System must choose between Consistency and Availability.")
        print("Simulating AP: Availability is maintained, but consistency may be lost.")
    else:
        print("No partition. System maintains both consistency and availability.")

def pacelc_scenario(network_partition=False):
    print("\n=== PACELC Theorem Scenario ===")
    if network_partition:
        print("P (Partition) => tradeoff between A (Availability) and C (Consistency).")
    else:
        print("E (Else) => tradeoff between L (Latency) and C (Consistency).")
        print("Example: DynamoDB chooses low latency (EL), Spanner chooses consistency (EC).")

# ========================
# Simulated Spanner-like Clock Sync
# ========================

def spanner_clock_sync():
    print("\n=== Simulated Spanner TrueTime API ===")
    gps_time = time.time()
    clock_drift = random.uniform(-0.001, 0.001)
    lower_bound = gps_time + clock_drift
    upper_bound = gps_time + clock_drift + 0.0001
    print(f"Time bounds: [{lower_bound:.6f}, {upper_bound:.6f}]")
    print("External consistency can be enforced using bounded uncertainty.")

# ========================
# Lab Simulation
# ========================

def run_distributed_db_simulation():
    print("=== Distributed Database Simulation ===")

    # 1. Raft Leader Election and Log Replication
    cluster = RaftCluster(num_nodes=5)
    cluster.elect_leader()
    cluster.replicate_log("INSERT INTO accounts VALUES (1, 'Alice', 1000)")
    cluster.replicate_log("UPDATE accounts SET balance = balance - 200 WHERE id = 1")
    cluster.show_logs()

    # 2. Sharding Example
    print("\n=== Sharding Simulation ===")
    sample_data = [{'id': i, 'name': f'User{i}'} for i in range(10)]
    shards = shard_data(sample_data, num_shards=3)
    for shard_id, records in shards.items():
        print(f"Shard {shard_id}: {records}")

    # 3. Replication of Shards
    print("\n=== Replication Simulation ===")
    replica_set = replicate_shards(shards)
    for shard_id, replicas in replica_set.items():
        print(f"Shard {shard_id} Replicas: {replicas}")

    # 4. CAP and PACELC Theorem Demonstrations
    cap_scenario(network_partition=True)
    pacelc_scenario(network_partition=False)

    # 5. Google Spanner Clock Synchronization
    spanner_clock_sync()

# Run the simulation
run_distributed_db_simulation()


=== Distributed Database Simulation ===

=== Leader Election Started ===
Node 3 becomes Leader with 5 votes (Term 1)

Leader (Node 3) is replicating command: 'INSERT INTO accounts VALUES (1, 'Alice', 1000)'
Replication complete.

Leader (Node 3) is replicating command: 'UPDATE accounts SET balance = balance - 200 WHERE id = 1'
Replication complete.

=== Logs of all Nodes ===
Node 0 Log: [(1, "INSERT INTO accounts VALUES (1, 'Alice', 1000)"), (1, 'UPDATE accounts SET balance = balance - 200 WHERE id = 1')]
Node 1 Log: [(1, "INSERT INTO accounts VALUES (1, 'Alice', 1000)"), (1, 'UPDATE accounts SET balance = balance - 200 WHERE id = 1')]
Node 2 Log: [(1, "INSERT INTO accounts VALUES (1, 'Alice', 1000)"), (1, 'UPDATE accounts SET balance = balance - 200 WHERE id = 1')]
Node 3 Log: [(1, "INSERT INTO accounts VALUES (1, 'Alice', 1000)"), (1, 'UPDATE accounts SET balance = balance - 200 WHERE id = 1')]
Node 4 Log: [(1, "INSERT INTO accounts VALUES (1, 'Alice', 1000)"), (1, 'UPDATE accounts S